# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


# Dates
start_date = "11-01-2021"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date
# update_date = "06-18-2025"

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "complete/"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

os.chdir(saved)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)

9697
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
19050


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,",",SRS17903639,False,NaN,Massachusetts
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,"Swab, Tracheal",SRS17903639,False,NaN,"USA: Massachusetts, Barnstable County"
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,",/",SRS17903636,False,NaN,Kentucky
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,"Swab Pool, Cloacal/Oropharyngeal",SRS17903636,False,NaN,"USA: Kentucky, Henderson County"
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,2023-06-07 02:01:28,1,22-005158-001,SRP441379,H5N1,NaN,SRS17903631,False,NaN,Maine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19045,SRR33830446,WGS,148.77,95232771,PRJNA1102327,SAMN48895426,Viral,37952018,USDA-NVSL,2025,...,2025-06-04 14:24:56,1,25-015677-004,SRP503016,NaN,"MILK, BULK TANK",SRS25266464,False,NaN,USA
19046,SRR33830447,WGS,148.43,106145246,PRJNA1102327,SAMN48895425,Viral,42182759,USDA-NVSL,2025,...,2025-06-04 14:25:01,1,25-015677-003,SRP503016,NaN,"MILK, BULK TANK",SRS25266463,False,NaN,USA
19047,SRR33830448,WGS,148.30,110812426,PRJNA1102327,SAMN48895424,Viral,43738503,USDA-NVSL,2025,...,2025-06-04 14:24:55,1,25-015677-002,SRP503016,NaN,"MILK, BULK TANK",SRS25266462,False,NaN,USA
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,2025-06-04 14:24:53,1,24-036379-001-tile,SRP503016,NaN,"MILK, BULK TANK",SRS25266461,False,NaN,USA


In [3]:
# Get list of genotypes

os.chdir(home + "references/")

genotypes_df = pd.read_excel("genotype_key.xlsx")

genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

               Run Assay Type  AvgSpotLen      Bases    BioProject  \
0      SRR24839058   AMPLICON      230.77   32942716   PRJNA980729   
1      SRR24839058   AMPLICON      230.77   32942716   PRJNA980729   
2      SRR24839059   AMPLICON      207.54   53993591   PRJNA980729   
3      SRR24839059   AMPLICON      207.54   53993591   PRJNA980729   
4      SRR24839060   AMPLICON      201.66   21703662   PRJNA980729   
...            ...        ...         ...        ...           ...   
19045  SRR33830446        WGS      148.77   95232771  PRJNA1102327   
19046  SRR33830447        WGS      148.43  106145246  PRJNA1102327   
19047  SRR33830448        WGS      148.30  110812426  PRJNA1102327   
19048  SRR33830449        WGS      131.52   67358891  PRJNA1102327   
19049  SRR33830450        WGS      131.13   69602732  PRJNA1102327   

          BioSample BioSampleModel     Bytes  \
0      SAMN35647642          Viral  18373327   
1      SAMN35647642          Viral  18373327   
2      SAMN3564

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,Massachusetts,SRR24839058,2025-05-09_10-45-20,SRR24839058.fa,A1,"NP:ea1, PB2:ea1, PB1:ea1, MP:ea1, NA:ea1, NS:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PB2, e...","99.87%, 99.80%, 99.91%, 99.90%, 99.79%, 99.88%...","2, 3, 2, 1, 3, 1, 6, 4",Ran on FASTA - No Coverage Report
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,"USA: Massachusetts, Barnstable County",SRR24839058,2025-05-09_10-45-20,SRR24839058.fa,A1,"NP:ea1, PB2:ea1, PB1:ea1, MP:ea1, NA:ea1, NS:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PB2, e...","99.87%, 99.80%, 99.91%, 99.90%, 99.79%, 99.88%...","2, 3, 2, 1, 3, 1, 6, 4",Ran on FASTA - No Coverage Report
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,Kentucky,SRR24839059,2025-05-09_10-45-06,SRR24839059.fa,A1,"NP:ea1, PA:ea1, MP:ea1, NA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PA, ea...","100.00%, 100.00%, 99.90%, 99.93%, 99.96%, 100....","0, 0, 1, 1, 1, 0, 1, 0",Ran on FASTA - No Coverage Report
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,"USA: Kentucky, Henderson County",SRR24839059,2025-05-09_10-45-06,SRR24839059.fa,A1,"NP:ea1, PA:ea1, MP:ea1, NA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PA, ea...","100.00%, 100.00%, 99.90%, 99.93%, 99.96%, 100....","0, 0, 1, 1, 1, 0, 1, 0",Ran on FASTA - No Coverage Report
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,Maine,SRR24839060,2025-05-09_10-45-39,SRR24839060.fa,A1,"NP:ea1, MP:ea1, NS:ea1, PA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:MP, ea...","99.80%, 99.90%, 99.88%, 99.72%, 99.65%, 99.66%...","3, 1, 1, 6, 8, 5, 6, 4",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19042,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,USA,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report
19043,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,USA,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report
19044,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,USA,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,USA,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report


In [5]:
# Get specific geolocation and name_state from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)


print(metadata["name_state"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

0                                Massachusetts
1        USA: Massachusetts, Barnstable County
2                                     Kentucky
3              USA: Kentucky, Henderson County
4                                        Maine
                         ...                  
19042                                      USA
19043                                      USA
19044                                      USA
19048                                      USA
19049                                      USA
Name: name_state, Length: 18082, dtype: object
18082


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,SRR24839058,2025-05-09_10-45-20,SRR24839058.fa,A1,"NP:ea1, PB2:ea1, PB1:ea1, MP:ea1, NA:ea1, NS:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PB2, e...","99.87%, 99.80%, 99.91%, 99.90%, 99.79%, 99.88%...","2, 3, 2, 1, 3, 1, 6, 4",Ran on FASTA - No Coverage Report,USA-MA
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,SRR24839058,2025-05-09_10-45-20,SRR24839058.fa,A1,"NP:ea1, PB2:ea1, PB1:ea1, MP:ea1, NA:ea1, NS:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PB2, e...","99.87%, 99.80%, 99.91%, 99.90%, 99.79%, 99.88%...","2, 3, 2, 1, 3, 1, 6, 4",Ran on FASTA - No Coverage Report,"USA: Massachusetts, Barnstable County"
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,SRR24839059,2025-05-09_10-45-06,SRR24839059.fa,A1,"NP:ea1, PA:ea1, MP:ea1, NA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PA, ea...","100.00%, 100.00%, 99.90%, 99.93%, 99.96%, 100....","0, 0, 1, 1, 1, 0, 1, 0",Ran on FASTA - No Coverage Report,USA-KY
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,SRR24839059,2025-05-09_10-45-06,SRR24839059.fa,A1,"NP:ea1, PA:ea1, MP:ea1, NA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PA, ea...","100.00%, 100.00%, 99.90%, 99.93%, 99.96%, 100....","0, 0, 1, 1, 1, 0, 1, 0",Ran on FASTA - No Coverage Report,"USA: Kentucky, Henderson County"
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,SRR24839060,2025-05-09_10-45-39,SRR24839060.fa,A1,"NP:ea1, MP:ea1, NS:ea1, PA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:MP, ea...","99.80%, 99.90%, 99.88%, 99.72%, 99.65%, 99.66%...","3, 1, 1, 6, 8, 5, 6, 4",Ran on FASTA - No Coverage Report,USA-ME
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19042,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report,USA
19043,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report,USA
19044,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report,USA
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report,USA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first

In [7]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# metadata.to_csv("metadata_genbank.csv")

In [ ]:
os.chdir(temp_files)
metadata = pd.read_csv("metadata_genbank_" + date_range + ".csv")

In [ ]:
# # # Upload saved data -- if doing this, make sure the above cell is commented out
# # os.chdir(temp_files + "saved/")
# # metadata_genbank = pd.read_csv("metadata_genbank.csv")
# # os.chdir(temp_files)

# # Get only updated dates

# # unknown_dates = metadata[(metadata["Collection_Date"] == "2024") | (metadata["Collection_Date"] == "2025")] # Dates we don't have
# years = ["2021", "2022", "2023", "2024", "2025"]
# unknown_dates = metadata[metadata["Collection_Date"].isin(years)]
# # known_dates = metadata[(metadata["Collection_Date"] != "2024") & (metadata["Collection_Date"] != "2025")] # Dates we've already gotten
# known_dates = metadata[~metadata["Collection_Date"].isin(years)]

# # Get new dates also 
# # new_dates = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible
# unknown_dates["Collection_Date"] = updated_unknown_dates

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# display(metadata)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6852c5913cf137baef0bf7dc&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
16-Mar-2024
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6852c59316506c9211018594&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6852c5963d1722948808c394&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6852c599c55ff0a00b07fa74&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_6852c59bf822d64931039b4c&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
20-Mar-2024
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_15188\4200550917.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unknown_dates["Collection_Date"] = updated_unknown_dates


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,SRR24839058,2025-05-09_10-45-20,SRR24839058.fa,A1,"NP:ea1, PB2:ea1, PB1:ea1, MP:ea1, NA:ea1, NS:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PB2, e...","99.87%, 99.80%, 99.91%, 99.90%, 99.79%, 99.88%...","2, 3, 2, 1, 3, 1, 6, 4",Ran on FASTA - No Coverage Report,USA-MA
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,SRR24839058,2025-05-09_10-45-20,SRR24839058.fa,A1,"NP:ea1, PB2:ea1, PB1:ea1, MP:ea1, NA:ea1, NS:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PB2, e...","99.87%, 99.80%, 99.91%, 99.90%, 99.79%, 99.88%...","2, 3, 2, 1, 3, 1, 6, 4",Ran on FASTA - No Coverage Report,"USA: Massachusetts, Barnstable County"
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,SRR24839059,2025-05-09_10-45-06,SRR24839059.fa,A1,"NP:ea1, PA:ea1, MP:ea1, NA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PA, ea...","100.00%, 100.00%, 99.90%, 99.93%, 99.96%, 100....","0, 0, 1, 1, 1, 0, 1, 0",Ran on FASTA - No Coverage Report,USA-KY
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,SRR24839059,2025-05-09_10-45-06,SRR24839059.fa,A1,"NP:ea1, PA:ea1, MP:ea1, NA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:PA, ea...","100.00%, 100.00%, 99.90%, 99.93%, 99.96%, 100....","0, 0, 1, 1, 1, 0, 1, 0",Ran on FASTA - No Coverage Report,"USA: Kentucky, Henderson County"
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,SRR24839060,2025-05-09_10-45-39,SRR24839060.fa,A1,"NP:ea1, MP:ea1, NS:ea1, PA:ea1, PB1:ea1, PB2:e...","ea1:22-003707-003:NP, ea1:22-003707-003:MP, ea...","99.80%, 99.90%, 99.88%, 99.72%, 99.65%, 99.66%...","3, 1, 1, 6, 8, 5, 6, 4",Ran on FASTA - No Coverage Report,USA-ME
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19042,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report,USA
19043,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report,USA
19044,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report,USA
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report,USA


In [ ]:


# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank_" + date_range + ".csv")

# metadata = metadata_genbank 

# display(metadata)

# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSample Accession,BioSampleModel,Bytes,Center Name,Collection_Date,...,is_retracted,isolate,isolation_source,name_state,retraction_detection_date_utc,sample,serotype,version,Host_Type,years
0,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,SRS17903639,Viral,18373327,United States Department of Agriculture,missing,...,False,A/Canada Goose/Massachusetts/22-005893-001/2022,",",Massachusetts,NaN,SRR24839058,H5N1,1,other,missing
1,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,SRS17903639,Viral,18373327,United States Department of Agriculture,missing,...,False,A/Canada Goose/Massachusetts/22-005893-001/2022,"Swab, Tracheal","USA: Massachusetts, Barnstable County",NaN,SRR24839058,H5N1,1,avian,missing
2,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,SRS17903636,Viral,29609119,United States Department of Agriculture,2022-02-08,...,False,A/Gadwall/Kentucky/22-005154-004/2022,",/",Kentucky,NaN,SRR24839059,H5N1,1,other,2022
3,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,SRS17903636,Viral,29609119,United States Department of Agriculture,2022-02-08,...,False,A/Gadwall/Kentucky/22-005154-004/2022,"Swab Pool, Cloacal/Oropharyngeal","USA: Kentucky, Henderson County",NaN,SRR24839059,H5N1,1,avian,2022
4,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,SRS17903631,Viral,10716272,United States Department of Agriculture,2022-02-17,...,False,A/Backyard bird/Maine/22-005158-001/2022,NaN,Maine,NaN,SRR24839060,H5N1,1,other,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18077,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,SRS25266467,Viral,39710063,USDA-NVSL,2025,...,False,25-015681-003,"MILK, BULK TANK",USA,NaN,SRR33830443,NaN,1,cattle,2025
18078,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,SRS25266466,Viral,51617990,USDA-NVSL,2025,...,False,25-015681-002,"MILK, BULK TANK",USA,NaN,SRR33830444,NaN,1,cattle,2025
18079,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,SRS25266465,Viral,45674405,USDA-NVSL,2025,...,False,25-015681-001,"MILK, BULK TANK",USA,NaN,SRR33830445,NaN,1,cattle,2025
18080,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,SRS25266461,Viral,26336550,USDA-NVSL,2024,...,False,24-036379-001-tile,"MILK, BULK TANK",USA,NaN,SRR33830449,NaN,1,cattle,2024


## Get host type

In [15]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['house-mouse', 'western gull', 'goat', 'rough-legged hawk', "cooper's hawk", 'corvus sp.', 'sanderling', 'guinea', 'sparrow', 'falcon', 'aves sp.', 'cattle milk product', 'anser sp.', 'ganada goose', 'partridge', 'snowy owl', 'dunlin', 'buteo jamaicensis', 'american wigeon', 'american black duck', 'grackle', 'mute swan', 'northern shoveler', 'strigiformes sp.', 'gallus gallus', 'anas rubripes', 'chukar', 'black-crowned night-heron', 'mareca strepera', 'common grackle', 'pelecanus erythrorhynchos', 'corvidae sp.', 'cackling goose', "geoffroy's cat", 'anser albifrons', 'black scoter', 'skunk', 'american crow', 'common merganser', 'dolphin', 'american coot', 'thalasseus maximus', 'cougar', 'antigone canadensis', 'melegris gallopavo', 'scoter', 'accipitridae sp.', 'great egret', 'american robin', 'coragyps atratus', 'brandt goose', 'mareca americana', 'barn owl', 'guinea fowl', 'lesser scaup', 'swallow', 'chicken', 'callipepla sp.', 'avian', 'larus argentatus', 'gavia immer', 'great black

In [16]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [18]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    
    if collection_date != collection_date: # If nan
        metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata.loc[num, "Collection_Date"] = collection_date
        else:
            try:
                parsed_date = dateutil.parser.parse(collection_date)
                date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
                metadata.loc[num, "Collection_Date"] = date
            except: # If no date at all
                metadata.loc[num, "Collection_Date"] = collection_date

    metadata = metadata.dropna(thresh=2)

# Make names

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["isolate"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata[["ReleaseDate", 'create_date', 'Collection_Date']])

,ReleaseDate,create_date,Collection_Date
0,2023-06-30,2023-06-07 02:01:47,missing
1,2023-06-30,2023-06-07 02:01:47,missing
2,2023-06-30,2023-06-07 02:01:22,2022-02-08
3,2023-06-30,2023-06-07 02:01:22,2022-02-08
4,2023-06-30,2023-06-07 02:01:28,2022-02-17
...,...,...,...
18077,2025-06-06,2025-06-04 14:25:00,2025
18078,2025-06-06,2025-06-04 14:25:01,2025
18079,2025-06-06,2025-06-04 14:24:54,2025
18080,2025-06-06,2025-06-04 14:24:53,2024


In [19]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [20]:
print(metadata)

      Assay Type  AvgSpotLen      Bases    BioProject     BioSample  \
0       AMPLICON      230.77   32942716   PRJNA980729  SAMN35647642   
2       AMPLICON      207.54   53993591   PRJNA980729  SAMN35647620   
4       AMPLICON      201.66   21703662   PRJNA980729  SAMN35647621   
6       AMPLICON      209.57   46055893   PRJNA980729  SAMN35647622   
8       AMPLICON      237.36   26794193   PRJNA980729  SAMN35647623   
...          ...         ...        ...           ...           ...   
18077        WGS      148.11  100597029  PRJNA1102327  SAMN48895429   
18078        WGS      148.50  131987248  PRJNA1102327  SAMN48895428   
18079        WGS      148.12  115677479  PRJNA1102327  SAMN48895427   
18080        WGS      131.52   67358891  PRJNA1102327  SAMN48895415   
18081        WGS      131.13   69602732  PRJNA1102327  SAMN48895414   

      BioSample Accession BioSampleModel     Bytes  \
0             SRS17903639          Viral  18373327   
2             SRS17903636          Vira

## Make FASTA files

In [21]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [ ]:
# print(fasta_files.keys())

In [22]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = originals + "complete/" + pair + "_" + date_range + "_andersen_updated.fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
>SRR24839171|A/./Massachusetts/A/Backyard bird/Massachusetts/22-009371-001/2022/2022|H5N1|USA-MA|2022-03-25|other|A1
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
>SRR24839298|A/./Nebraska/A/Swan/Nebraska/22-008227-001/2022/2022|H5N1|USA-NE|2022-03-13|other|A1
nan
nan
nan
nan
nan
nan
nan
nan
>SRR24839316|A/./Kansas/A/Backyard bird/Kansas/22-008114-001/2022/2022|H5N1|USA-KS|2022-03-16|other|A1
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
>SRR24839413|A/./Texas/A/Pheasant/Texas/22-010008-002/2022/2022|H5N1|USA-TX|2022-04-01|other|A1
nan
nan
>SRR24839416|A/./Texas/A/Pheasant/

## De-Duplication

In [ ]:
# De-duplication 

# Gisaid 
dfs_gisaid_list = []
for genotype in genotypes:

    # gisaid = downloads + "GISAID/complete/" + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "_Antarctica_North_America_South_America/"
    gisaid = downloads + "GISAID/complete/all_genotypes/11-01-2021--06-13-2025_all_genotypes_Antarctica_North_America_South_America/"
    # gisaid = downloads + "Cats/Datasets/GISAID/"

    os.chdir(gisaid)

    dfs_gisaid = create_dataframes(gisaid)
    dfs_gisaid_list.append(dfs_gisaid)

dfs_gisaid = {}
for df_gisaid in dfs_gisaid_list:
    dfs_gisaid = dfs_gisaid | df_gisaid
# dfs_gisaid2 = create_dataframes(gisaid2)

A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
C2.1_HA
C2.1_MP
C2.1_NA
C2.1_NP
C2.1_NS
C2.1_PA
C2.1_PB1
C2.1_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
Minor100_HA
Minor100_MP
Minor100_NA
Minor100_NP
Minor100_NS
Minor100_PA
Minor100_PB1
Minor100_PB2
Minor94_HA
Minor94_MP
Minor94_NA
Minor94_NP
Minor94_NS
Minor94_PA
Minor94_PB1
Minor94_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
C2

In [ ]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(originals + "complete/")

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_HA
A3_MP
A3_MP
A3_NA
A3_NA
A3_NP
A3_NP
A3_NS
A3_NS
A3_PA
A3_PA
A3_PB1
A3_PB1
A3_PB2
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_

In [ ]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [ ]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [ ]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

80
968


In [ ]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

0
1
  isolate_partial                                        full_header  \
0        23OS0170  >EPI_ISL_19873848|A/wood_duck/Ohio/23OS0170/20...   

                                            sequence  
0  atggagaacatagtactacttcttgcaatagttagccttgttaaaa...  
len full df: 1
Keeping nothing:  1
len deduplicated: 1
0
1
  isolate_partial                                        full_header  \
0        23OS0170  >EPI_ISL_19873848|A/wood_duck/Ohio/23OS0170/20...   

                                            sequence  
0  atgagtcttctaaccgaggtcgaaacgtacgttctctctatcgtcc...  
len full df: 1
Keeping nothing:  1
len deduplicated: 1
0
1
  isolate_partial                                        full_header  \
0        23OS0170  >EPI_ISL_19873848|A/wood_duck/Ohio/23OS0170/20...   

                                            sequence  
0  atgaatccaaatcaaaagataacaaccattggatcaatctgtatgg...  
len full df: 1
Keeping nothing:  1
len deduplicated: 1
0
1
  isolate_partial                                     

In [ ]:
# # If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)
#             full_dfs[key].append(dataframes[i])

## Create FASTA files combining Andersen and GISAID

In [ ]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + date_range + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
C2.1_HA
C2.1_MP
C2.1_NA
C2.1_NP
C2.1_NS
C2.1_PA
C2.1_PB1
C2.1_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
Minor100_HA
Minor100_MP
Minor100_NA
Minor100_NP
Minor100_NS
Minor100_PA
Minor100_PB1
Minor100_PB2
Minor94_HA
Minor94_MP
Minor94_NA
Minor94_NP
Minor94_NS
Minor94_PA
Minor94_PB1
Minor94_PB2


: 